In [11]:
import pandas as pd
import joblib
import scipy.sparse as sp
import numpy as np

ridge = joblib.load("../models/ridge_baseline.joblib")
print("Ridge model loaded successfully, alpha =", ridge.alpha)

Ridge model loaded successfully, alpha = 5.0


In [12]:
# Load remaining saved artifacts
name_vec = joblib.load("../models/name_vectorizer.joblib")
desc_vec = joblib.load("../models/desc_vectorizer.joblib")
scaler   = joblib.load("../models/scaler.joblib")
ohe      = joblib.load("../models/onehot_encoder.joblib")

# Load validation data
val = pd.read_csv("../data/processed/val.csv")
val["name"] = val["name"].fillna("")
val["item_description"] = val["item_description"].fillna("")

NUM_COLS = ["item_condition_id","shipping","category_depth","name_length","desc_length",
            "name_word_count","has_description","is_branded","cat_avg_price","brand_avg_price"]
CAT_COLS = ["main_category","sub_category","sub_sub_category","condition_label"]
for c in CAT_COLS:
    val[c] = val[c].fillna("missing")

# Transform only -- these were already fitted on train in train_ridge.py
X_name_val = name_vec.transform(val["name"])
X_desc_val = desc_vec.transform(val["item_description"])
X_num_val = scaler.transform(val[NUM_COLS])
X_cat_val = ohe.transform(val[CAT_COLS])

X_val = sp.hstack([X_name_val, X_desc_val, sp.csr_matrix(X_num_val), X_cat_val]).tocsr()


print("X_val shape:", X_val.shape)

X_val shape: (9945, 25759)


In [13]:
ridge.predict(X_val)

array([3.10573566, 2.88928125, 2.21488466, ..., 2.51781581, 3.52460409,
       2.59732849], shape=(9945,))